In [4]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.1-flash-lite",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [6]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is exploring fictional or hypothetical information regarding the lunar city of Lunapolis, specifically focusing on its environment, population, and socio-political climate.\n\n## SUMMARY\n\nThe following details about the fictional city of Lunapolis have been established:\n- **Environment:** Extreme temperature fluctuations ranging from -100C to 120C with clear skies.\n- **Population:** Approximately 100,000 residents identified as "cheese miners."\n- **Political Climate:** There is a high probability of a cheese miners\' union strike driven by dissatisfaction with the current city president.\n\n## ARTIFACTS\n\nNone.\n\n## NEXT STEPS\n\nContinue responding to user inquiries regarding the fictional status and developments of Lunapolis and its inhabitants.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='f340eb47-f937-4467-bcdb-bbd8ad074555'),
 

In [7]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

The user is exploring fictional or hypothetical information regarding the lunar city of Lunapolis, specifically focusing on its environment, population, and socio-political climate.

## SUMMARY

The following details about the fictional city of Lunapolis have been established:
- **Environment:** Extreme temperature fluctuations ranging from -100C to 120C with clear skies.
- **Population:** Approximately 100,000 residents identified as "cheese miners."
- **Political Climate:** There is a high probability of a cheese miners' union strike driven by dissatisfaction with the current city president.

## ARTIFACTS

None.

## NEXT STEPS

Continue responding to user inquiries regarding the fictional status and developments of Lunapolis and its inhabitants.


## Trim/delete messages

In [8]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [9]:
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [10]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='4ac69a21-6594-4f50-9e05-951daafc115f'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='bbef72ae-5dc2-49b7-aae0-7b744a85fc2b', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='53087519-299e-4376-9eb0-8f477bf0d7af'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='6809338b-936a-446b-b9a1-d838e79681d2', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='d4bc6b15-eb27-405d-b840-768a0b916229'),
              AIMessage(content=[{'type': 'text', 'text': "I am an AI, so I don't have physical

In [11]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': "I am an AI, so I don't have physical access to your device to feel its temperature, but **you should check it yourself carefully.**\n\nIf the device is **extremely hot** to the touch, please **unplug it immediately** and let it cool down before attempting any further troubleshooting. It may be overheating, which can trigger safety mechanisms that prevent it from powering on.\n\n**If the device is at a normal temperature, could you please tell me:**\n\n1.  **What kind of device is it?** (e.g., laptop, phone, gaming console, kitchen appliance).\n2.  **Does it show any lights at all?** (e.g., a charging light, a power button LED, or screen backlight flickering).\n3.  **Did anything happen right before it stopped working?** (e.g., did it fall, spill liquid, or was there a power surge?).\n\nKnowing these details will help me give you more specific steps to fix it.", 'extras': {'signature': 'EnMKcQFpFH0TPC/pGhSjkoPdhQOiE3isPq1v0OXqTQhXG2f/jk6ywPisa2BuXvvYh5/zYalOrX